# DP9 スモーク3：凍結の前の最後の確認（Qwen2.5-7B-Instruct・L4）

> ## 🔑 先に許可しておくシークレット
> - **`GITHUB_TOKEN`（必須）**
> - `HF_TOKEN`（任意。なくても動きます）
>
> 左の鍵のマークで「ノートブックからのアクセス」をオンにしてから「すべてのセルを実行」してね。

**これは何？**：新鮮な目のレビューで「凍結の前に確かめて」と言われた 3 つを測ります。**ベンチマークの問題は使わず、採点もしません。** 使うのは、これまでと同じ C5-A の 2 つの文と、スモークのために書いた算数 4 問だけです（記録は数字だけ）。

1. **確率 10⁻⁶ の語に「合う位置」がどれくらいあるか**（Qwen は自信の強い歩が多いので、9 割を割るおそれ）
2. **4 種類 × 32 本 = 128 行をまとめて計算しても、1 本ずつと同じ答えになるか**（M1 の本番の大きさ）と、いちばん長いときのメモリ
3. **本当に終わるまで引いたときの、1 つの位置の実際の時間**（見積もりの ×1.7 が合っているか）

**使い方**
1. ランタイム → ランタイムのタイプを変更 → **L4**（ハイメモリは不要）。**新しいランタイム**で始めてね（前のモデルが残っていると、メモリが足りなくなります。念のため、セル 3 でも片付けます）。
2. **「すべてのセルを実行」**。**25〜45 分くらい（約 0.7〜1.3 CU）** です。
3. 結果は Drive の `DECISION_POINTS/dp9/SMOKE3_<時刻>/SMOKE.json`（数字だけ）。終わったら Claude に知らせてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

- 通知の頭には 🚀（始まり）／⚠（エラー）／🎉（終わり）が付きます。

**止まる場所**：セル 1〜3（取得・インストール・照合・読み込み）。B の M1 とメモリ、C の時間は、合わなくても止めずに記録します。

In [ ]:
# Cell 1 — Drive, GPU check, the exact head, install, pin the model revision (Hub API) and download it to local disk.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'DP9 スモーク3（Qwen2.5-7B・L4・合う位置・128行・実際の時間）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'GodPork - Pork DP9 smoke3', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'DP9_SMOKE3'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('⚠ 止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

import tempfile
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # allocation only (as DP4-DP8)
drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), "GPU のランタイムにしてね（L4）"
GPU_NAME = gpu.stdout.strip().split(",")[0]
print("GPU:", gpu.stdout.strip())
if "L4" not in GPU_NAME:
    print("⚠️ このランタイムは L4 ではありません（" + GPU_NAME + "）。動きますが、測った時間は L4 の値になりません。")

EXPECTED_HEAD = "34c6515cb198e0cd11725582f9ca146f7b0298e0"
BRANCH = "decision-points"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
THEME = Path("/content/drive/MyDrive/DECISION_POINTS")
STAMP = dt.datetime.now(dt.timezone(dt.timedelta(hours=9))).strftime("%Y%m%d_%H%M%S")
OUT = THEME / "dp9" / ("SMOKE3_" + STAMP)
OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = Path("/content/qwen2.5-7b-instruct")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        return None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN") or get_secret("GH_TOKEN")
assert token, "Colab のシークレットに GITHUB_TOKEN を登録して、ノートブックからのアクセスをオンにしてね"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
print("exact head:", head)

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "transformers==4.57.6",
                    "huggingface_hub", "pyarrow", "scikit-learn", "scipy"], capture_output=True, text=True)
(OUT / "install_log.txt").write_text(r.stdout + "\n--- STDERR ---\n" + r.stderr)
assert r.returncode == 0, r.stderr[-4000:]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import transformers, torch
assert transformers.__version__ == "4.57.6", transformers.__version__
from pork import dp9_smoke as S

def atomic_json(p, value):
    p.parent.mkdir(parents=True, exist_ok=True)
    data = json.dumps(value, ensure_ascii=False, indent=1, sort_keys=True) + "\n"
    fd, tmp = tempfile.mkstemp(prefix=p.name + ".", suffix=".tmp", dir=p.parent)
    with os.fdopen(fd, "w", encoding="utf-8") as f:
        f.write(data); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, p)

SMOKE = {"schema": S.SMOKE_SCHEMA, "code_commit": EXPECTED_HEAD, "gpu_name": GPU_NAME, "gpu": gpu.stdout.strip(),
         "model_id": S.MODEL_ID, "dtype": S.DTYPE, "started_jst": STAMP, "python": sys.version.split()[0],
         "torch": torch.__version__, "transformers": transformers.__version__}

# The model revision: the Hub's current commit of the main branch, recorded here; the DP9 gate will pin it.
from huggingface_hub import HfApi, snapshot_download
HF = get_secret("HF_TOKEN")
SMOKE["hf_token_used"] = bool(HF)
info = HfApi().model_info(S.MODEL_ID, token=HF, files_metadata=True)
REVISION = info.sha
def _lfs_sha(sib):
    lfs = getattr(sib, "lfs", None)
    if lfs is None:
        return None
    return lfs.get("sha256") if isinstance(lfs, dict) else getattr(lfs, "sha256", None)
FILES = {s.rfilename: {"size": s.size, "lfs_sha256": _lfs_sha(s)} for s in info.siblings}
t0 = time.time()
snapshot_download(S.MODEL_ID, revision=REVISION, local_dir=str(MODEL_DIR), token=HF,
                  allow_patterns=["*.json", "*.safetensors", "merges.txt", "vocab.json", "tokenizer*"])
SMOKE["download_seconds"] = time.time() - t0
t0 = time.time()
checked = {}
for name, meta in sorted(FILES.items()):
    p = MODEL_DIR / name
    if not p.exists():
        continue
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""):
            h.update(chunk)
    checked[name] = {"size": p.stat().st_size, "sha256": h.hexdigest(),
                     "matches_hub_lfs": None if meta["lfs_sha256"] is None else h.hexdigest() == meta["lfs_sha256"]}
assert all(v["matches_hub_lfs"] is not False for v in checked.values()), "ダウンロードしたファイルが Hub の記録と違います"
SMOKE["model_revision"] = REVISION
SMOKE["model_files"] = checked
SMOKE["hash_seconds"] = time.time() - t0
CONFIG = json.loads((MODEL_DIR / "config.json").read_text())
SMOKE["config"] = {k: CONFIG.get(k) for k in ("architectures", "vocab_size", "hidden_size", "num_hidden_layers",
                                               "num_attention_heads", "num_key_value_heads", "max_position_embeddings",
                                               "tie_word_embeddings", "use_sliding_window", "torch_dtype")}
atomic_json(OUT / "SMOKE.json", SMOKE)
print("model revision:", REVISION, "| files:", len(checked), "| download", round(SMOKE["download_seconds"]), "s")

In [ ]:
# Cell 2 — GPU vs plain-Python implementations on synthetic logits with ties (vocab 2,000 and Qwen's output size): the
# base path's per-step signals (DP1's step_signals), pure sampling at T 1.0, the alt first-token draw (base token
# removed) and the w_tau choice (closest to 1e-6; top-1 and excluded ids never chosen).  No model.
import math, random, struct
import torch as torch_mod
from pork import pork_t_v1 as pt
from pork import q4b_samplers as qb
from pork import dp1_forks as dp
from pork import dp8_bridge as d8
Bg = pt.backend("torch", "cuda")
V_OUT = int(CONFIG["vocab_size"])

def f32(x):
    return struct.unpack("f", struct.pack("f", x))[0]

def synth(rng, vocab):
    xs = [f32(rng.gauss(0.0, rng.choice((1.0, 2.5, 4.0)))) for _ in range(vocab)]
    for i in range(0, vocab, 97):
        xs[i] = xs[0]
    if rng.random() < 0.3:
        j = rng.randrange(vocab); xs[j] = f32(xs[j] + 12.0)
    if rng.random() < 0.2:
        xs[1] = xs[0] = max(xs)
    if rng.random() < 0.4:
        m = max(xs)
        v = f32(m + math.log(d8.TAU) + rng.choice((0.0, 0.3, -0.3)))
        for j in rng.sample(range(2, vocab), 3):
            xs[j] = v
    return xs

rng = random.Random(S.PARITY_SEED)
names = ("SIGNALS", "PURE", "ALT", "TAU")
total = {k: 0 for k in names}; mism = {k: 0 for k in names}
t0 = time.time()
for vocab, steps in ((2000, 150), (V_OUT, 8)):
    mg = mr = None
    for step in range(steps):
        xs = synth(rng, vocab)
        g = torch_mod.tensor(xs, dtype=torch_mod.float32, device="cuda")
        a, b = dp.step_signals(Bg, g, mg), dp.ref_step_signals(xs, mr)
        total["SIGNALS"] += 1
        if ((a["top1"], a["top2"]) != (b["top1"], b["top2"]) or abs(a["p1"] - b["p1"]) > 1e-9
                or abs(a["p2"] - b["p2"]) > 1e-9 or abs(a["entropy"] - b["entropy"]) > 1e-9):
            mism["SIGNALS"] += 1
        mg, mr = a["mu_after"], b["mu_after"]
        u = rng.random()
        total["PURE"] += 1
        if tuple(int(v) for v in qb.pure_sample(Bg, g, u, 1.0)) != tuple(int(v) for v in qb.ref_pure(xs, u, 1.0)):
            mism["PURE"] += 1
        exa = b["top1"] if step % 2 == 0 else rng.randrange(vocab)
        u2 = rng.random() if step % 7 else 0.9999999999
        aa, ab = d8.alt_sample(Bg, g, u2, 1.0, exa), d8.ref_alt(xs, u2, 1.0, exa)
        total["ALT"] += 1
        if aa != ab or aa == exa:
            mism["ALT"] += 1
        ex = rng.sample(range(vocab), 4)
        ta, tb = d8.tau_choice(Bg, g, b["top1"], ex), d8.ref_tau_choice(xs, b["top1"], ex)
        total["TAU"] += 1
        if (ta["token"], ta["matched"]) != (tb["token"], tb["matched"]) or abs(ta["p"] - tb["p"]) > 1e-9 * tb["p"]:
            mism["TAU"] += 1
SMOKE["parity"] = {"total": total, "mismatches": mism, "vocab_sizes": [2000, V_OUT], "seconds": time.time() - t0,
                   "pass": all(v == 0 for v in mism.values())}
atomic_json(OUT / "SMOKE.json", SMOKE)
print(json.dumps(SMOKE["parity"], indent=1))
assert SMOKE["parity"]["pass"], "サイン・サンプリングの照合が合いません — Claude に見せてね"
print("PARITY PASS")

In [ ]:
# Cell 3 — load the model (bfloat16; the library's default attention for Qwen2 is recorded), and check the tokenizer:
# EOS ids, the excluded ids for w_tau, the chat template (counts only), digit tokens.
from pork import kv_cached_backend as kv
from pork import dp2_real_forks as d2
from pork import dp8_runner as rr
from pork.ct1_runner import torch_fns
from pork.ultimapork_completion_c5a import PROMPTS as C5A_PROMPTS
import gc
for _old in ("model", "_single", "_batch"):
    globals().pop(_old, None)   # a model left by an earlier run in this runtime
gc.collect(); torch_mod.cuda.empty_cache()
t0 = time.time()
tokenizer = transformers.AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = transformers.AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), torch_dtype=torch_mod.bfloat16).to("cuda")
model.eval()
SMOKE["load_seconds"] = time.time() - t0
ATTN = model.config._attn_implementation
def set_attn(name):
    try:
        if hasattr(model, "set_attn_implementation"):
            model.set_attn_implementation(name)
        else:
            model.config._attn_implementation = name
    except Exception:
        model.config._attn_implementation = name
    return model.config._attn_implementation == name
EOS = kv.eos_ids(tokenizer, model)
TAU_EXCLUDED = rr.excluded_ids(tokenizer, EOS)
VOCAB = int(model.get_output_embeddings().weight.shape[0])
DEVICE = "cuda"
_single, _batch = torch_fns(model, torch_mod, DEVICE)
Bt = pt.backend("torch", DEVICE)
decode = lambda ids: tokenizer.decode(list(ids), skip_special_tokens=True)
im_start = tokenizer.convert_tokens_to_ids("<|im_start|>")
c5a_ids = [kv.chat_prompt_ids(tokenizer, t) for _, t in C5A_PROMPTS]
own_ids = [kv.chat_prompt_ids(tokenizer, d2.user_message(q)) for q, _ in S.OWN_PROBLEMS]
digits_single = all(len(tokenizer.encode(c, add_special_tokens=False)) == 1 for c in "0123456789")
n12345 = len(tokenizer.encode("12345", add_special_tokens=False))
digit_tokens = sum(d2.has_digit(tokenizer.decode([i])) for i in range(len(tokenizer)))
SMOKE["model"] = {
    "attn_implementation": ATTN, "dtype": str(model.dtype), "vocab_out": VOCAB, "tokenizer_len": len(tokenizer),
    "eos_ids": list(EOS), "tau_excluded_count": len(TAU_EXCLUDED), "special_ids_count": len(tokenizer.all_special_ids),
    "chat_turn_starts_c5a": [sum(int(x == im_start) for x in ids) for ids in c5a_ids],
    "default_system_turn": all(sum(int(x == im_start) for x in ids) == 3 for ids in c5a_ids),
    "prompt_tokens_c5a": [len(x) for x in c5a_ids], "prompt_tokens_own": [len(x) for x in own_ids],
    "digits_are_single_tokens": digits_single, "tokens_in_12345": n12345, "vocab_tokens_with_digit": digit_tokens,
    "gpu_gb_total": torch_mod.cuda.get_device_properties(0).total_memory / 1e9,
    "gpu_gb_after_load": torch_mod.cuda.memory_allocated() / 1e9,
}
assert str(model.dtype) == "torch.bfloat16" and VOCAB == V_OUT
assert all(0 <= x < VOCAB for x in TAU_EXCLUDED)
SMOKE["workers_possible"] = max(1, int((SMOKE["model"]["gpu_gb_total"] - 2.0) // (SMOKE["model"]["gpu_gb_after_load"] + 4.0)))
atomic_json(OUT / "SMOKE.json", SMOKE)
print(json.dumps(SMOKE["model"], indent=1), "\nworkers possible on this GPU:", SMOKE["workers_possible"])

In [ ]:
# Cell 4 — A. The four problems written for the smoke: greedy base paths (DP8's generator, max 512 tokens), then at
# the stride-8 and stride-16 positions before the answer: matched for w_tau (DP7's rule: within +-0.5 decades of
# 1e-6), and the r bands.  Counts only.
SMOKE["schema"] = "dp9-smoke3-v1"
BASES, COV = [], []
for (q, ans), ids in zip(S.OWN_PROBLEMS, own_ids):
    gen = rr.generate_base_tau(Bt, _single(), ids, EOS, TAU_EXCLUDED, d2.MAX_NEW_TOKENS)
    rec = dp.base_record(gen["token_ids"], gen["eos"], decode, "#### " + ans)
    a = int(rec["answer_token"] or 0)
    BASES.append({"ids": tuple(ids), "gen": gen, "a": a})
    row = {"answer_token": a, "tokens": len(gen["token_ids"]), "correct": rec["correct"],
           "prompt_ids_sha256": hashlib.sha256(json.dumps(list(ids)).encode()).hexdigest()}
    for stride in (8, 16):
        pos = list(range(0, a, stride))
        m = [bool(gen["steps"][t]["tau_matched"]) for t in pos]
        r = [1.0 - float(gen["steps"][t]["p1"]) for t in pos]
        row["s" + str(stride)] = {"positions": len(pos), "matched": sum(m),
                                  "r_le_1e-4": sum(v <= 1e-4 for v in r), "r_gt_0.05": sum(v > 0.05 for v in r),
                                  "unmatched_with_r_le_1e-4": sum((not x) and v <= 1e-4 for x, v in zip(m, r))}
    COV.append(row)
for stride in (8, 16):
    tot = sum(r_["s" + str(stride)]["positions"] for r_ in COV)
    SMOKE.setdefault("coverage", {})["stride%d_matched_share" % stride] = (
        sum(r_["s" + str(stride)]["matched"] for r_ in COV) / max(1, tot))
SMOKE["coverage"]["per_problem"] = COV
atomic_json(OUT / "SMOKE.json", SMOKE)
print(json.dumps(SMOKE["coverage"], indent=1))

In [ ]:
# Cell 5 — B. M1 at the real size: at the middle position t of the first problem's path, one one-row prefill of
# prompt + base[:t], repeated to 128 rows: keep (w1) x 32, swap (w2) x 32, alt (32 own draws, w1 excluded) x 32,
# swapt (w_tau) x 32.  Each group's first row vs that prefix run alone (teacher forced, 32 steps; >= 29 of 32, as B1);
# rows that share the group's first token are fed the same reference tokens, the other alt rows their own argmax.
# Then the memory peak with the longest prompt + 512 tokens at 128 and 160 rows (4 decode steps).  Recorded only.
sync = torch_mod.cuda.synchronize
def is_oom(e):
    return isinstance(e, torch_mod.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()
def one_row(prefix):
    with torch_mod.no_grad():
        o = model(input_ids=torch_mod.tensor([list(prefix)], device=DEVICE), use_cache=True, logits_to_keep=1)
    return o.past_key_values, o.logits[0, -1].float()

pb = BASES[0]
toks, steps_ = pb["gen"]["token_ids"], pb["gen"]["steps"]
t = pb["a"] // 2
prefix = pb["ids"] + tuple(toks[:t])
cache1, lg1 = one_row(prefix)
w1, w2, wt = int(toks[t]), int(steps_[t]["top2"]), int(steps_[t]["tau_token"])
alts = [int(d8.alt_sample(Bt, lg1, d8.stream("alt", 0, 0, j)[0], 1.0, w1)) for j in range(32)]
firsts = [w1] * 32 + [w2] * 32 + alts + [wt] * 32
reps = [0, 32, 64, 96]
refs = {}
for r0 in reps:
    st, past, new, seqr = _single(), None, prefix + (firsts[r0],), []
    for _ in range(32):
        lgr, past = st(new, past)
        tk = int(lgr.argmax()); seqr.append(tk); new = (tk,)
    refs[r0] = seqr
del past
big, how = S.expand_cache(cache1, 128)
bst, past = _batch(), big
rows = [(f,) for f in firsts]
agree = {r0: 0 for r0 in reps}
for s_ in range(32):
    lgb, past = bst(rows, past)
    am = [int(v) for v in lgb.argmax(dim=-1).tolist()]
    for r0 in reps:
        agree[r0] += int(am[r0] == refs[r0][s_])
    rows = [((refs[reps[r // 32]][s_],) if firsts[r] == firsts[reps[r // 32]] else (am[r],)) for r in range(128)]
del big, past, lgb, cache1
torch_mod.cuda.empty_cache()
M1_128 = {"position": t, "expand_method": how, "agree_steps": [agree[r0] for r0 in reps],
          "groups": ["keep", "swap", "alt_row0", "swapt"], "alt_distinct_first_tokens": len(set(alts)),
          "pass": all(v >= 29 for v in agree.values())}
filler = tuple(t_ for b in BASES for t_ in b["gen"]["token_ids"]) * 3
longest = max(own_ids + c5a_ids, key=len)
MEM = {"prompt_tokens": len(longest)}
for n in (128, 160):
    torch_mod.cuda.empty_cache(); torch_mod.cuda.reset_peak_memory_stats()
    try:
        pc, plg = one_row(tuple(longest) + filler[:508])
        bigm, _m = S.expand_cache(pc, n)
        del pc
        S.generate_merged(Bt, _batch(), bigm, [int(plg.argmax())] * n, [d8.stream("keep", 0, 0, j % 32) for j in range(n)],
                          [0] * n, (-1,), 1.0, 4)
        MEM[str(n)] = torch_mod.cuda.max_memory_allocated() / 1e9
        del bigm
    except Exception as e:
        if not is_oom(e):
            raise
        MEM[str(n)] = None
        pc = bigm = None
    torch_mod.cuda.empty_cache()
MEM["tokens_at_peak"] = len(longest) + 512
SMOKE["m1_128"] = M1_128
SMOKE["memory_max_length_gb"] = MEM
atomic_json(OUT / "SMOKE.json", SMOKE)
print(json.dumps({"m1_128": M1_128, "memory": MEM}, indent=1))

In [ ]:
# Cell 6 — C. Whole positions, as the main run would compute them (merged way), with real EOS: for each of the four
# problems, positions t = 0, a/3, 2a/3: one one-row prefill, 32 alt draws, 128 rows (keep, swap, alt, swapt) sampled
# to EOS (at most 511 - t continuation tokens), then a free batch of 32 rows (first token drawn from the same logits,
# w1 allowed).  Recorded: wall time, the longest continuation, rows that hit the length limit, and the time the
# per-step model of the second smoke predicts with L = the answer position (pork/dp9_smoke.SMOKE2_PER).
import random
P2 = S.SMOKE2_PER
FREE_STREAMS = lambda row, i: [tuple(random.Random(202610171290 + 1000003 * (row + 1) + 10007 * (i + 1) + j).random()
                                     for _ in range(512)) for j in range(32)]
WALL = []
for row, pb in enumerate(BASES):
    toks, steps_, a = pb["gen"]["token_ids"], pb["gen"]["steps"], pb["a"]
    for i, t in enumerate(sorted({0, a // 3, (2 * a) // 3})):
        if t >= len(toks):
            continue
        sync(); t0 = time.time()
        cache1, lg1 = one_row(pb["ids"] + tuple(toks[:t]))
        w1, w2, wt = int(toks[t]), int(steps_[t]["top2"]), int(steps_[t]["tau_token"])
        st_alt = [d8.stream("alt", row, i, j) for j in range(32)]
        alts = [int(d8.alt_sample(Bt, lg1, st_alt[j][0], 1.0, w1)) for j in range(32)]
        firsts = [w1] * 32 + [w2] * 32 + alts + [wt] * 32
        streams = ([d8.stream("keep", row, i, j) for j in range(32)] + [d8.stream("swap", row, i, j) for j in range(32)]
                   + st_alt + [d8.stream("swapt", row, i, j) for j in range(32)])
        offs = [0] * 64 + [1] * 32 + [0] * 32
        big, _m = S.expand_cache(cache1, 128)
        outs = S.generate_merged(Bt, _batch(), big, firsts, streams, offs, EOS, 1.0, 511 - t)
        del big
        sync(); t_main = time.time() - t0
        t1 = time.time()
        fs = FREE_STREAMS(row, i)
        ffirst = [int(qb.pure_sample(Bt, lg1, fs[j][0], 1.0)[0]) for j in range(32)]
        b32, _m = S.expand_cache(cache1, 32)
        fouts = S.generate_merged(Bt, _batch(), b32, ffirst, fs, [1] * 32, EOS, 1.0, 511 - t)
        del b32, cache1
        torch_mod.cuda.empty_cache()
        sync(); t_free = time.time() - t1
        lens = [len(o["token_ids"]) for o in outs]
        flens = [len(o["token_ids"]) for o in fouts]
        pred_main = P2["PREFILL1"] + P2["EXPAND"]["128"] + max(0, a - t) * P2["DECODE"]["128"]
        pred_free = P2["EXPAND"]["32"] + max(0, a - t) * P2["DECODE"]["32"]
        WALL.append({"problem": row, "t": t, "a": a, "seconds_main": t_main, "seconds_free": t_free,
                     "longest_main": max(lens), "longest_free": max(flens), "median_main": sorted(lens)[64],
                     "hit_limit_main": sum(1 for o in outs if not o["eos"] and len(o["token_ids"]) >= 511 - t),
                     "hit_limit_free": sum(1 for o in fouts if not o["eos"] and len(o["token_ids"]) >= 511 - t),
                     "predicted_main": pred_main, "predicted_free": pred_free})
        print(f"problem {row} t={t}: main {t_main:.1f}s (model {pred_main:.1f}s), free {t_free:.1f}s (model {pred_free:.1f}s), "
              f"longest {max(lens)}")
tot_wall = sum(w["seconds_main"] + w["seconds_free"] for w in WALL)
tot_pred = sum(w["predicted_main"] + w["predicted_free"] for w in WALL)
SMOKE["wall"] = {"positions": WALL, "total_seconds": tot_wall, "total_predicted": tot_pred,
                 "ratio_actual_to_model": tot_wall / tot_pred if tot_pred else None}
atomic_json(OUT / "SMOKE.json", SMOKE)
print("actual / model =", round(SMOKE["wall"]["ratio_actual_to_model"], 2))

In [ ]:
# Cell 7 — projection of the DP9 main run (40 items, one worker) with the measured ratio, for the owner's choice
# (4 kinds x 32 rows x stride 8, free 32 rows at every third position) and the fallback (stride 16).  Then done.
ratio = SMOKE["wall"]["ratio_actual_to_model"]
PROJ = []
for stride in (8, 16):
    for L in (240, 300):
        raw = 40 * S.per_item_merged_free(P2, 4, 32, stride, L) / 3600
        PROJ.append({"stride": stride, "L": L, "hours_raw": raw, "hours_measured_ratio": raw * ratio,
                     "cu_measured_ratio": raw * ratio * S.CU_PER_L4_HOUR, "hours_x2_7": raw * 2.7})
SMOKE["projection3"] = PROJ
SMOKE["finished_jst"] = dt.datetime.now(dt.timezone(dt.timedelta(hours=9))).strftime("%Y%m%d_%H%M%S")
atomic_json(OUT / "SMOKE.json", SMOKE)
for p_ in PROJ:
    print(f"stride {p_['stride']:>2}  L {p_['L']}:  {p_['hours_measured_ratio']:.1f} h ({p_['cu_measured_ratio']:.0f} CU) "
          f"with the measured ratio x{ratio:.2f};  x2.7 -> {p_['hours_x2_7']:.1f} h")
print("✅ スモーク3 おわり。結果：DECISION_POINTS/dp9/SMOKE3_" + STAMP + "/SMOKE.json")
notify('🎉 終わりました：DP9 スモーク3（合う位置・128行・実際の時間）。結果は dp9/SMOKE3_' + STAMP, tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()